# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Two paper findings and my methodology questions

**Finding A:** ___ (one sentence, in my own words, from the paper)
- **Question I would ask:** Where does the label for this finding come from, and is it measured after the window the features come from?
- **Why it matters:** If the label overlaps the feature window, the result could be partly built in.
- **Why I think the authors made this choice:** ___ (for example, a broad audience, or disclosed limits)

**Finding B:** ___ (one sentence, in my own words)
- **Question I would ask:** Does the validation design (split by ___) support the claim for pages or clients the model hasn't seen?
- **Why it matters:** A random split can reward memorizing one client's pages.
- **What I'd like to see next:** ___ (for example, results by client or by month)

I'm asking these as practice at the next level of rigor, the way I'd want my own work reviewed. I'm not grading the paper.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

FEATS = ["___feat1___", "___feat2___", "___feat3___", "___feat4___", "___feat5___"]
LABEL, GROUP = "___label___", "___group_col___"
d = df.dropna(subset=FEATS + [LABEL]).reset_index(drop=True)

def precision_at_k(y, s, k=50):
    return float(np.asarray(y)[np.argsort(-np.asarray(s))[:k]].mean())

def fit_eval(train, test):
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
    m.fit(train[FEATS], train[LABEL])
    s = m.predict_proba(test[FEATS])[:, 1]
    return round(precision_at_k(test[LABEL], s, 50), 3), round(roc_auc_score(test[LABEL], s), 3)

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# BEFORE: random row split (groups can appear in both train and test)
tr_a, te_a = train_test_split(d, test_size=0.3, random_state=0)
before = fit_eval(tr_a, te_a)

# AFTER: grouped split (no group in both sets)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
tr_i, te_i = next(gss.split(d, groups=d[GROUP]))
after = fit_eval(d.iloc[tr_i], d.iloc[te_i])

pd.DataFrame(
    {"Split": ["Random rows (before)", "Grouped by " + GROUP + " (after)"],
     "Precision@50": [before[0], after[0]],
     "AUC": [before[1], after[1]]}
)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
DATE = "___date_col___"
cut = d[DATE].quantile(0.7)
tr_t, te_t = d[d[DATE] <= cut], d[d[DATE] > cut]
print("Time-aware (train earlier, test later):", fit_eval(tr_t, te_t))

The random split gave Precision@50 of ___ and the grouped split gave ___. The drop of ___ shows how much the weaker split flattered the model. I report the grouped number as my honest result.

In [ ]:
# Check 1: single-feature strength. A very high AUC from one feature is a warning sign.
rows = []
for f in FEATS:
    rows.append({"feature": f, "single_feature_AUC": round(roc_auc_score(d[LABEL], d[f].rank()), 3)})
pd.DataFrame(rows).sort_values("single_feature_AUC", ascending=False)

In [ ]:
# Check 2: correlation of each feature with the label
d[FEATS + [LABEL]].corr(numeric_only=True)[LABEL].drop(LABEL).round(3).sort_values(ascending=False)

In [ ]:
# Check 3: group overlap in the honest split (should be 0)
print(len(set(d.iloc[tr_i][GROUP]) & set(d.iloc[te_i][GROUP])))

## 3) Leakage audit

| Feature | When is it measured? | Could it overlap the label window? | Verdict |
|---|---|---|---|
| ___ | ___ | ___ | OK / suspect |
| ___ | ___ | ___ | OK / suspect |
| ___ | ___ | ___ | OK / suspect |
| ___ | ___ | ___ | OK / suspect |
| ___ | ___ | ___ | OK / suspect |

- No feature had a suspiciously high single-feature AUC (highest was ___). *If one did, say which and what you did about it.*
- Group overlap between train and test is 0.
- The label window starts after the features end: ___ (explain how you know).

In [ ]:
m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
tr, te = d.iloc[tr_i], d.iloc[te_i].copy()
m.fit(tr[FEATS], tr[LABEL])
te["score"] = m.predict_proba(te[FEATS])[:, 1]

top = te.sort_values("score", ascending=False).head(50)
print("False positives in top 50:", int((top[LABEL] == 0).sum()))
top[top[LABEL] == 0][[GROUP] + FEATS + ["score"]].head(5)

In [ ]:
missed = te[(te[LABEL] == 1)].sort_values("score").head(5)
missed[[GROUP] + FEATS + ["score"]]

**Failure patterns:**
- False positives tend to ___.
- Missed positives tend to ___.

## 4) Claim rewrite

| My original claim | Problem | Safer version |
|---|---|---|
| "The model predicts which pages need a refresh." | A proxy label, not proof of need | "In this sample, the model ranks pages that carry the proxy label higher than a simple rule does (observed, directional)." |
| "Feature ___ drives performance." | Importance describes the model, not causes | "Feature ___ is associated with the label in this sample (measured)." |
| "The model works on any site." | One sample, grouped split only | "Results are measured on ___ groups and may not hold elsewhere." |
| ___ | ___ | ___ |

## 5) Self-check

- [x] Two paper findings, each with a constructive methodology question
- [x] Before/after under an honest split, with the numbers shown
- [x] Leakage audit with a verdict per feature
- [x] Real error examples
- [x] Claims rewritten with observed / measured / directional / decision-support language
- [x] No client names, domains, URLs or tokens in any output

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.